### 1. OpenAI Chat Completions API 구조
- **메시지 역할(Role)**: `system` (기본 지시사항), `user` (사용자 입력), `assistant` (LLM의 응답)
- **주요 파라미터**: `temperature` (응답의 무작위성 제어, ReAct에서는 0에 가깝게 설정하는 것이 유리), `max_tokens` 등
- **API 호출 구조**: `openai` 파이썬 패키지를 이용한 기본 호출 방법 소개

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

client  = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
# test
response = client.chat.completions.create(
    model='gpt-5.4-nano',
    messages=[{'role':'user','content':'ReAct agent에 대해서 한줄로 설명해줘'}],
    temperature=0,    
    max_completion_tokens = 100
)
print(response.choices[0].message.content)
print(f'토큰사용량 : {response.usage}')

ReAct 에이전트는 **추론(Reasoning)과 행동(Acting)을 번갈아 수행**하며, 필요할 때 도구를 호출해 문제를 단계적으로 해결하는 에이전트입니다.
토큰사용량 : CompletionUsage(completion_tokens=50, prompt_tokens=17, total_tokens=67, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


In [2]:
response.usage.total_tokens

67

### 2. ReAct 시스템 프롬프트 설계
시스템 프롬프트에는 다음이 반드시 포함되어야 합니다:
- 에이전트의 역할 지정
- 사용 가능한 도구(Tool) 목록과 그 사용법
- 반드시 따라야 하는 출력 형식 (Thought -> Action -> Observation 패턴)

### 3. ReAct 루프 직접 구현
- **while 또는 for 루프 기반 에이전트 구조**: 에이전트가 `Finish` 액션을 낼 때까지 계속해서 응답을 생성하고 도구를 실행하는 과정을 반복합니다.
- **종료 조건**: 모델의 출력에서 `Action: Finish`가 감지되면 무한 루프를 종료하고 최종 답을 반환합니다.
- **최대 반복 횟수 제한**: 무한 루프를 방지하기 위해 최대 스텝 수를 지정합니다.

In [ ]:
import re
class SimpleReactAgent:    
    def __init__(self,model = 'gpt-5.4-nano', max_steps=5):        
        self.model = model
        self.max_steps = max_steps
        self.tools = {}
        self.systemp_prompt = ''
    def register_tool(self, name,func,description):
        self.tools[name] = {'func':func, 'description':description}
    def _build_system_prompt(self):
        tool_desc = '\n'.join([ f"- {name}:{info['description']}" for name, info in self.tools.items()  ])
        self.systemp_prompt = f'''당신은 주어진 질문에 정확하게 답변하기 위해 도구를 사용하는 AI 에이전트입니다.
        사용 가능한 도구:
        {tool_desc}

        반드시 다음 형식을 따라 응답하세요:
        Thought: [현재 상황에 대한 추론]
        Action: [도구이름][입력값]

        도구 실행결과는 Observation으로 제공됩니다
        최종 답변을 제출할 때는 반드시 다음 형식을 사용하세요:
        Thought: [최종 추론]
        Action: Finish[최종 답변]

        중요: 한 번에 하나의 Action만 수행하세요
        '''
    def _parse_action(self, text):
        match = re.search(r'Action\s*:\s*(\w+)\[(.+?)\]', text, re.DOTALL)
        if match:
            return match.group(1).strip(), match.group(2).strip()
        return None, None
    def _execute_tool(self, tool_name, tool_input):
        if tool_name == 'Finish':
            return None
        elif tool_name in self.tools:
            try:
                return self.tools[tool_name]['func'](tool_input)
            except Exception as e:
                return f'도구 실행 오류:{e}'
    def run(self, question):
        self._build_system_prompt()
        messages = [
            {'role':'system','content':self.systemp_prompt},
            {'role':'user','content':f'Question:{question}'}
        ]
        print(f'Question:{question}')
        print('='*60)
        for step in range(self.max_steps):
            response = client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0
            )
            assistant_msg = response.choices[0].message.content
            messages.append({'role':'assistant','content':assistant_msg})

            print(f'\n -- Step {step+1} --')
            print(assistant_msg)

            action_name, action_input =  self._parse_action(assistant_msg)

            if action_name =='Finish':
                print(f"\n{'='*60}")
                print(f'final Answer : {action_input}')                
                return action_input
            elif action_name:
                observation = self._execute_tool(action_name,action_input)
                obs_msg = f'Observation: {observation}'
                messages.append({'role':'user','content':obs_msg})
                print(f'\n{obs_msg}')
            else:
                messages.append({'role':'user', 'content':'형식오류:Action:도구이름[입력값] 형식으로 응답해주세요'})
        print('\n최대 단계수에 도달했습니다.')
        return None

In [4]:
# Define simple tools
def calculator(expression):
    allowed = set('0123456789+-*/.() ')
    if all(c in allowed for c in expression):
        return str(eval(expression))
    return "허용되지 않는 수식입니다."

def knowledge_base(query):
    kb = {
        "서울 인구": "서울특별시의 인구는 약 950만 명입니다 (2024년 기준).",
        "서울 면적": "서울특별시의 면적은 약 605.2 km2 입니다.",
        "한국 GDP": "대한민국의 GDP는 약 1조 7천억 달러입니다 (2024년 기준).",
        "한국 인구": "대한민국의 총 인구는 약 5,170만 명입니다 (2024년 기준).",
        "파이썬": "Python은 1991년 귀도 반 로섬이 만든 프로그래밍 언어입니다.",
        "인공지능": "인공지능(AI)은 인간의 지능을 모방하는 컴퓨터 시스템을 연구하는 분야입니다."
    }
    query_lower = query.lower()
    for key, value in kb.items():
        if key in query_lower or any(word in query_lower for word in key.split()):
            return value
    return f"'{query}'에 대한 정보를 찾을 수 없습니다."

In [5]:
agent = SimpleReactAgent()
agent.register_tool('Calc',calculator,'수학 수식을 계산합니다. 예 calculator[2+3]')
agent.register_tool('knowbs',knowledge_base,'내장 지식베이스에서 정보를 검색합니다. 예 knowledge_base[서울 인구]')
result = agent.run('서울의 인구와 면적을 조회하고, 인구밀도(인구/면적)을 계산해주세요')


Observation: 서울특별시의 인구는 약 950만 명입니다 (2024년 기준).


BadRequestError: Error code: 400 - {'error': {'message': "Invalid type for 'messages[3].content[0]': expected an object, but got a string instead.", 'type': 'invalid_request_error', 'param': 'messages[3].content[0]', 'code': 'invalid_type'}}